# ChessMind – training the v1 model

---

## 1. Goal and scope

- Build a lightweight position evaluation function (eval) correlated with Stockfish scores, intended for shallow search (depth 1–3).
- Clear workflow: CSV shards → training → save the best model → play test.
- v1 deliberately limits feature and architecture complexity so that runs are fast, repeatable, and easy to diagnose.

---

## 2. Data and split

- Input data: CSV shards containing FEN and target centipawn (cp) scores computed by Stockfish.
- Phase proportions: 30% opening (OPEN), 30% middlegame (MID), 40% endgame (END) — keeps endgames visible during learning.
- Validation is separated at the shard level; no mixing of positions between the training and validation sets.
- Sample character: mostly “typical” positions (balanced material, no immediate combinations); extreme positions (mate in 1, large material swings) are less frequent.

---

## 3. Position representation (v1)

- General features: side to move, game phase indicator, simple material features (sum for each side and the material difference).
- Technical features present in some runs: engine depth/time from the data generator; these are “meta” signals not directly tied to geometry.
- Intentional omissions in v1: no full 8×8 board geometry (explicit piece locations), no explicit pawn structure or king safety. This keeps v1 fast, but its view of the position is simplified.

---

## 4. Model and training process

- Architecture: a shallow multilayer perceptron (MLP); a single scalar prediction (eval).
- Optimization: typical setup with AdamW; number of epochs on the order of tens; the “best checkpoint” is saved using a validation metric.
- Metrics observed in logs: steady loss decrease at the beginning, flattening later; per-epoch time stays within a narrow range, indicating a steady data stream.

---

## 5. Validation and interpretation

- Monitored values: train_loss, val_loss, val_MAE_cp, epoch time, and number of validation positions.
- Typical validation result: MAE around 120 cp; good correlation in the mid-range, larger spread at the extremes (very high and very low scores).
- “Pred vs Target (val)” plot: a dense band near the diagonal in the center; dispersion increases towards extremes — expected for a simplified representation.

---

## 6. Using the evaluation in play

- Playing algorithm: shallow negamax (1–3 plies) that uses the model’s prediction as the leaf evaluation.
- Move order: based on the library’s standard enumeration of legal moves; no extra geometry-driven preferences.
- Characteristic behavior in v1: when several moves receive similar scores, the choice is sensitive to iteration order and tiny prediction differences, which may lead to repetitive move patterns.

---

## 7. Observations from test games (based on logs)

- partia.txt (continuous mode, Black): frequent knight trips to the rim (e.g., Nh6) and back (Ng8), plus repeated rook shuffling between g8 and h8. No clear developmental progress; oscillatory moves around the king.
- partia_z_czlowiekiem.txt (live game): persistent “back-and-forth” rook motif (Rg8 ↔ Rh8) while White is active. The log also shows invalid SAN/uci inputs on the opponent side — this concerns the testing interface, not the model itself.
- partia_z_czlowiekiem_ale_inny_case.txt (mid-game start position): quick and correct recognition of a ready mating motif “Qxh2#”. In obvious tactical spots, shallow depth can be sufficient.
- partia_z_czlowiekiem_v2.txt (longer game): natural start; in the middlegame, reduced sensitivity to king safety and piece coordination; in the ending, simple defensive sequences against mating threats are not always detected.
- partia_koncowka.txt (K+Q vs K, White): after a series of checks comes “Qg6+?”, then “Kxg6” and a draw by insufficient mating material. This illustrates that in forcing endgames, control over exchanges may be insufficient despite material advantage.
- partia_koncowka_mat_w_1.txt (mate in 1 with rooks): instead of delivering the immediate mate, rooks are shuffled cyclically; the number of plies grows without real progress on the board.

---

## 8. Known limitations in v1

- Limited spatial information: without directly encoding piece locations on the 64 squares it is harder to differentiate between moves that look similar by material.
- Flat decisions: when several moves have close scores, the choice depends heavily on move enumeration order and tiny prediction differences.
- Endgame behavior: in forcing sequences, control over exchanges is not always maintained, which can reduce a previously gained advantage.
- Plot interpretation: greater spread at cp extremes translates to lower precision on extreme positions.

---

## 9. Glossary and interpretation notes

- Centipawn (cp): evaluation unit; 100 cp ≈ value of a pawn.
- MAE [cp]: mean absolute error between the model prediction and the target; easy to interpret in practice.
- FEN: textual chess position representation used in shards and testing interfaces.
- Training logs: include the number of shards, device (cpu/cuda), per-epoch metrics, and information about saving the best model.
- Play logs: include an ASCII board, FEN, SAN/uci moves, and response time; they can also show messages about invalid opponent inputs from the interface.

---

## 10. Summary

- v1 serves as a lightweight evaluation function for quick experiments with shallow search.
- In tests it combines the ability to spot simple tactical motifs with predictable limitations stemming from a frugal position representation.
- The phenomena observed in games (piece oscillations, missing mate in 1, sacrificing the queen in a simple ending) are consistent with the scope and assumptions of v1.

---

## 11. Plots

![Loss Curve](../../plots/model_v1/loss_curve.png)
![Predicted vs Target](../../plots/model_v1/pred_vs_target.png)
![Residual Histogram](../../plots/model_v1/residual_hist.png)
